# Dependency and copy number — `dependency.csv` and `copy_number.csv`

Verifies tables 8 and 9 — the two functional-evidence layers built from the augmented `data/augmented/` files (`15_CRISPRGeneEffect.csv`, `16_OmicsCNGeneWGS.csv`). Both files key their gene columns by symbol text (Entrez-style `SYMBOL (EntrezID)`), resolved onto `gene_reference` via `joins.build_symbol_bridge` — this notebook checks the resolved values look like the real biology already characterised in `eda/individual_eda/15_CRISPRGeneEffect.ipynb` and `16_OmicsCNGeneWGS.ipynb`, not that the join logic re-derives it from scratch.

In [ ]:
import pandas as pd

### Dependency score — should center near 0 with a negative tail; `RPL3` should be strongly negative
More negative = a cell line needs that gene more (Chronos convention, confirmed in Phase 1's EDA). `RPL3` (a core ribosomal protein) was the sign-convention spot-check gene there — it should be strongly negative here too.

In [ ]:
gene_reference = pd.read_csv("../../data/processed/gene_reference.csv")
rpl3_id = gene_reference.loc[gene_reference["symbol"] == "RPL3", "ensembl_id"].iloc[0]
print(f"RPL3 -> {rpl3_id}")

In [ ]:
dep_sample_parts, rpl3_parts = [], []
for chunk in pd.read_csv("../../data/processed/dependency.csv", chunksize=5_000_000):
    dep_sample_parts.append(chunk["dependency_score"].sample(min(50_000, len(chunk)), random_state=1))
    rpl3_parts.append(chunk[chunk["ensembl_id"] == rpl3_id])

dep_sample = pd.concat(dep_sample_parts)
rpl3 = pd.concat(rpl3_parts)
print(dep_sample.describe())
print(f"\nRPL3: n={len(rpl3)}, mean={rpl3['dependency_score'].mean():.3f}, median={rpl3['dependency_score'].median():.3f}")

**Confirmed:** overall sample mean -0.145 (matches Phase 1's df15 EDA finding of -0.149), centered near 0 with a long negative tail (min -5.16 in this sample). `RPL3` — median **-2.608**, matching Phase 1's `15_CRISPRGeneEffect.ipynb` finding **exactly** (same median to three decimals) — the symbol-bridge join preserved the value correctly end to end.

### Copy number — should center near 1.0 (diploid baseline)
Confirmed in Phase 1's `16_OmicsCNGeneWGS.ipynb` as this file's own linear-scale baseline (not the textbook 2.0 — this file's own median sits near 1.0). `amp`/`del` are descriptive-only calls at `>1.5`/`<0.5` around that baseline — the real scoring threshold belongs to `scoring/`, not here.

In [ ]:
cn_sample_parts = []
amp_n = del_n = total_n = 0
for chunk in pd.read_csv("../../data/processed/copy_number.csv", chunksize=5_000_000):
    cn_sample_parts.append(chunk["copy_number"].sample(min(50_000, len(chunk)), random_state=1))
    amp_n += chunk["amp"].sum()
    del_n += chunk["del"].sum()
    total_n += len(chunk)

cn_sample = pd.concat(cn_sample_parts)
print(cn_sample.describe())
print(f"\namp rate: {amp_n/total_n*100:.2f}%  ({amp_n:,} of {total_n:,})")
print(f"del rate: {del_n/total_n*100:.2f}%")

**Confirmed:** sample mean 1.05, median ~1.02 — right where Phase 1's EDA found this file's diploid baseline to sit. `amp` (>1.5) fires on 5.81% of rows, `del` (<0.5) on 1.54% — both a small, plausible minority, consistent with most gene-line pairs being copy-neutral.

### Verdict
Both tables reproduce the exact values Phase 1's per-file EDA already established (RPL3's median matches to three decimal places), confirming the Entrez-symbol bridge carried values through correctly. No concerns found.